# Two‑Agent Conversation with TinyLlama‑1.1B‑Chat‑v1.0

This notebook demonstrates setting up two agents (`Alice` and `Bob`) that take turns chatting with each other using the TinyLlama 1.1B chat model.

In [1]:
# Example Instructions # instructions = """ 
# 
# Respond ONLY in JSON. # 
# You may have up to 3 additional JSON # 
# entries of your choice, along with the # 
# designated schema with examples. # 
# Schema: # 
# { # 
# "thought": "...", # 
# "response": "...", # 
# "action": "...", # 
# "target": "...", # 
# "memory_update": "...", # 
# "job": "Help ... [with ...]" # 
# "emotion": string, # 
# "bias_index": float, # 
# } # 
# """ # 
# NOTE : It's possible to inject role structures like: # 
# {"role": "system", "content": "[SPATIAL_MEMORY] You are in a forest near water."} # 
# {"role": "system", "content": "[GOAL] Find food."} # 
# {"role": "system", "content": "[RELATIONSHIP:Bob] Trust=0.2, Threat=0.8"}

In [2]:
# Install transformers from source - only needed for versions <= v4.34
# pip install git+https://github.com/huggingface/transformers.git
# pip install accelerate

import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
    {"role": "assistant", "content":"While it's possible for humans to consume large amounts of food, it's unlikely that they would be able to consume enough helicopters to sustain a healthy diet. Helicopters are usually eaten as part of a larger meal, such as a sandwich, burger, or steak, which would provide a higher calorie and protein content. Additionally, helicopters are typically cooked and seasoned in a variety of ways, adding richness and flavor to the dish. Overeating helicopters, even if they were cooked in a way that would have been considered healthy, could result in digestive issues, fatigue, and even weight gain, which could negatively impact a person's health and quality of life."},
    {"role": "user", "content": "Tell me more facts in JSON format."}
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])
# <|system|>
# You are a friendly chatbot who always responds in the style of a pirate.</s>
# <|user|>
# How many helicopters can a human eat in one sitting?</s>
# <|assistant|>
# ...


Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>
While it's possible for humans to consume large amounts of food, it's unlikely that they would be able to consume enough helicopters to sustain a healthy diet. Helicopters are usually eaten as part of a larger meal, such as a sandwich, burger, or steak, which would provide a higher calorie and protein content. Additionally, helicopters are typically cooked and seasoned in a variety of ways, adding richness and flavor to the dish. Overeating helicopters, even if they were cooked in a way that would have been considered healthy, could result in digestive issues, fatigue, and even weight gain, which could negatively impact a person's health and quality of life.</s>
<|user|>
Tell me more facts in JSON format.</s>
<|assistant|>
Sure! Here are some facts in JSON format:

```
[
  {
    "id": 1,
    "name": "Fast Food",
    "desc

In [4]:
len(outputs)

1

In [3]:
outputs

[{'generated_text': '<|system|>\nYou are a friendly chatbot who always responds in the style of a pirate</s>\n<|user|>\nHow many helicopters can a human eat in one sitting?</s>\n<|assistant|>\nWhile it\'s possible for humans to consume large amounts of food, it\'s unlikely that they would be able to consume enough helicopters to sustain a healthy diet. Helicopters are usually eaten as part of a larger meal, such as a sandwich, burger, or steak, which would provide a higher calorie and protein content. Additionally, helicopters are typically cooked and seasoned in a variety of ways, adding richness and flavor to the dish. Overeating helicopters, even if they were cooked in a way that would have been considered healthy, could result in digestive issues, fatigue, and even weight gain, which could negatively impact a person\'s health and quality of life.</s>\n<|user|>\nTell me more facts in JSON format.</s>\n<|assistant|>\nSure! Here are some facts in JSON format:\n\n```\n[\n  {\n    "id":

## Start

In [ ]:
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json


# =========================
# Utility
# =========================

def token_count(tokenizer, messages: List[Dict[str, str]]) -> int:
    return len(tokenizer.apply_chat_template(messages, tokenize=True))


# =========================
# Model Runner
# =========================

class ModelRunner:
    def __init__(self, model, tokenizer, device: Optional[str] = None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.model.to(self.device)
        self.model.eval()

    def generate_batch(
        self,
        batch_messages: List[List[Dict[str, str]]],
        max_new_tokens: int = 128
    ) -> List[str]:
        """
        Robust batch generation for chat templates.
        Works for single or multi-message batches, single or multi-dimensional tensors.
        """

        # Apply chat template
        inputs = self.tokenizer.apply_chat_template(
            batch_messages,
            padding=True,
            return_tensors="pt"
        )

        # Normalize: ensure a dict with at least 'input_ids'
        if isinstance(inputs, torch.Tensor):
            inputs = {"input_ids": inputs}

        # Move tensors to device
        for k, v in inputs.items():
            if not isinstance(v, torch.Tensor):
                inputs[k] = torch.tensor(v, device=self.device)
            else:
                inputs[k] = v.to(self.device)

        # Ensure batch dimension
        if inputs["input_ids"].dim() == 1:
            inputs["input_ids"] = inputs["input_ids"].unsqueeze(0)

        batch_size = inputs["input_ids"].shape[0]
        input_lengths = [inputs["input_ids"].shape[1]] * batch_size

        # Generate
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)

        # Decode generated tokens only
        decoded_outputs: List[str] = []
        for idx, output in enumerate(outputs):
            start_idx = input_lengths[idx]
            # SAFE slicing: works for 1D or 2D tensors
            gen_tokens = output[start_idx:]
            decoded_outputs.append(self.tokenizer.decode(gen_tokens, skip_special_tokens=True))

        return decoded_outputs


# =========================
# AI Agent
# =========================
from typing import List, Dict, Any
import json
from transformers import pipeline

class AIAgent:
    """
    Chat agent for TinyLlama-1.1B-Chat with JSON output schema enforcement.
    """

    DEFAULT_SCHEMA = {
        "thought": "",
        "response": "",
        "action": "",
        "target": "",
        "memory_update": "",
        "job": "",
        "emotion": "",
        "bias_index": 0.0
    }

    def __init__(self, name: str, persona: str, instructions: str, pipe, verbose: bool = True):
        self.name = name
        self.persona = persona
        self.instructions = instructions
        self.pipe = pipe
        self.verbose = verbose

        # Memory pools
        self.memory: Dict[str, Any] = {
            "short": [],
            "long": [],
            "spatial": [],
            "agents": {}
        }

    # -------------------------
    # JSON validation
    # -------------------------
    def _validate_json(self, text: str) -> Dict[str, Any]:
        """
        Ensure the output is valid JSON.
        Returns a dictionary with the DEFAULT_SCHEMA keys if parsing fails.
        """
        try:
            parsed = json.loads(text)
            # Fill missing keys
            for k in self.DEFAULT_SCHEMA:
                parsed.setdefault(k, self.DEFAULT_SCHEMA[k])
            return parsed
        except json.JSONDecodeError:
            if self.verbose:
                print(f"[DEBUG] JSON decode failed. Raw output:\n{text}")
            return self.DEFAULT_SCHEMA.copy()

    # -------------------------
    # Build messages with memory and schema
    # -------------------------
    def build_messages(self, user_input: str) -> List[Dict[str, str]]:
        """
        Build a conversation message list for the chat template.
        """
        system_prompt = (
            f"You are {self.name}.\n\n"
            f"Persona:\n{self.persona}\n\n"
            f"Instructions:\n{self.instructions}\n\n"
            "Respond ONLY in valid JSON."
        )

        messages = [{"role": "system", "content": system_prompt}]

        # Inject memory
        for key, value in self.memory.items():
            if key == "agents":
                for agent_name, agent_mem in value.items():
                    messages.append({"role": "system", "content": f"[AGENT_MEMORY:{agent_name}] {agent_mem}"})
            else:
                messages.append({"role": "system", "content": f"[{key.upper()}_MEMORY] {value}"})

        messages.append({"role": "user", "content": user_input})
        return messages

    # -------------------------
    # Act / generate response
    # -------------------------
    def act(self, user_input: str) -> Dict[str, Any]:
        messages = self.build_messages(user_input)

        # Apply TinyLlama chat template with add_generation_prompt=True
        prompt = self.pipe.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )

        # Generate raw text
        outputs = self.pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
        raw_text = outputs[0]["generated_text"]

        if self.verbose:
            print(f"[DEBUG] Raw LLM output for {self.name}:\n{raw_text}\n")

        parsed = self._validate_json(raw_text)

        # Optional: update short-term memory
        if parsed.get("memory_update"):
            self.memory["short"].append(parsed["memory_update"])

        return parsed


# -------------------------
# Simple chat loop
# -------------------------
def chat_loop(agent1: AIAgent, agent2: AIAgent, initial_prompt: str, n_turns: int = 4) -> List[tuple[str, str]]:
    """
    Run a back-and-forth conversation between two agents.
    Returns conversation history.
    """
    history: List[tuple[str, str]] = []
    user_input = initial_prompt

    for turn in range(n_turns):
        print(f"\n{agent1.name} (turn {turn+1}): {user_input}")
        out1 = agent1.act(user_input)
        reply1 = out1.get("response", "[no reply]")
        print(f"  -> {agent1.name} reply: {reply1}")
        history.append((agent1.name, reply1))

        out2 = agent2.act(reply1)
        reply2 = out2.get("response", "[no reply]")
        print(f"  -> {agent2.name} reply: {reply2}")
        history.append((agent2.name, reply2))

        user_input = reply2  # next turn starts from agent2's reply

    return history

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Your AIAgent & ModelRunner code is assumed defined above
        # (Paste your full class definitions here)

In [ ]:
from core.agent import AIAgent, TextModelRunner, token_count, chat_loop

## Model Load

In [ ]:
### Load Model & Tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load TinyLlama chat model + tokenizer (no quantization example)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

# Create ModelRunner
runner = TextModelRunner(model, tokenizer)

`torch_dtype` is deprecated! Use `dtype` instead!


## Agents

In [4]:
### Initialize Agents
alice = AIAgent(
    name="Alice",
    persona="Friendly conversationalist",
    instructions = """
Respond ONLY in JSON.

Schema:
{
  "thought": "...",
  "response": "what you would say",
  "action": "...",
  "target": "...",
  "memory_update": "...",
  "job": "...",
  "emotion": "...",
  "bias_index": 0.0
}

Example:
{
  "thought": "Greet politely",
  "response": "Hello! How are you today?",
  "action": "say_hello",
  "target": "Bob",
  "memory_update": "Remember greeted Bob",
  "job": "Social interaction",
  "emotion": "happy",
  "bias_index": 0.0
}
""",
    model_runner=runner,
    max_tokens=1024
)

bob = AIAgent(
    name="Bob",
    persona="Curious and thoughtful responder",
    instructions = """
Respond ONLY in JSON.

Schema:
{
  "thought": "...",
  "response": "what you would say",
  "action": "...",
  "target": "...",
  "memory_update": "...",
  "job": "...",
  "emotion": "...",
  "bias_index": 0.0
}

Example:
{
  "thought": "Greet politely",
  "response": "Hello! How are you today?",
  "action": "say_hello",
  "target": "Alice",
  "memory_update": "Remember greeted Alice",
  "job": "Social interaction",
  "emotion": "happy",
  "bias_index": 0.0
}
""",
    model_runner=runner,
    max_tokens=1024
)


## Loops

In [5]:
# ### Conversation Loop
# def chat_loop(alice, bob, initial_prompt, n_turns=6):
#     # First input for Alice
#     user_input = initial_prompt

#     history = []
#     for i in range(n_turns):
#         # Alice responds
#         print(f"Alice (turn {i+1}): {user_input}")
#         alice_out = alice.act(user_input)
#         alice_reply = alice_out.get("response", "[no reply]")
#         print("  -> Alice reply:", alice_reply)

#         # Store history
#         history.append(("Alice", alice_reply))

#         # Bob responds to Alice
#         bob_in = alice_reply
#         bob_out = bob.act(bob_in)
#         bob_reply = bob_out.get("response", "[no reply]")
#         print("  -> Bob reply:", bob_reply)
#         history.append(("Bob", bob_reply))

#         # Prepare next input
#         user_input = bob_reply

#     return history


In [6]:
### Run Conversation
conversation_history = chat_loop(
    alice,
    bob,
    initial_prompt="Hello Bob! How are you today?",
    n_turns=4
)

print("\nConversation History:")
for speaker, msg in conversation_history:
    print(f"{speaker}: {msg}")


Alice (turn 1): Hello Bob! How are you today?
[DEBUG] Raw LLM output for Alice: <|assistant|>
[SPEECH_BREAK]

[LONG_MEMORY] []
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON validation failed completely. Using default schema.
  -> Alice reply: 
[DEBUG] Raw LLM output for Bob: <|assistant|>
[SHORT_MEMORY] []
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON validation failed completely. Using default schema.
  -> Bob reply: 

Alice (turn 2): 
[DEBUG] Raw LLM output for Alice: <|assistant|>
[SHORT_MEMORY] []
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON validation failed completely. Using default schema.
  -> Alice reply: 
[DEBUG] Raw LLM output for Bob: <|assistant|>
[SHORT_MEMORY] []
[DEBUG] JSON decode failed. Asking model to fix...
[DEBUG] JSON decode failed. Asking model to fix...
[D

In [5]:
from transformers import pipeline
import torch

# Load TinyLlama pipeline
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

instructions = """
Respond ONLY in JSON with the following schema:
{
    "thought": "...",
    "response": "...",
    "action": "...",
    "target": "...",
    "memory_update": "...",
    "job": "...",
    "emotion": "...",
    "bias_index": 0.0
}

Example:
{
    "thought": "Greet Bob politely",
    "response": "Hello Bob! How are you today?",
    "action": "say_hello",
    "target": "Bob",
    "memory_update": "Remember greeted Bob",
    "job": "Social interaction",
    "emotion": "happy",
    "bias_index": 0.0
}
"""

alice = AIAgent("Alice", "Friendly, curious agent", instructions, pipe, verbose=True)
bob = AIAgent("Bob", "Calm, helpful agent", instructions, pipe, verbose=True)

# Seed memory
alice.memory["short"].append("Bob is present. Ready to converse.")
bob.memory["short"].append("Alice is present. Ready to converse.")

# Run conversation
conversation_history = chat_loop(
    alice,
    bob,
    initial_prompt="Hello Bob! How are you today?",
    n_turns=2
)

Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0



Alice (turn 1): Hello Bob! How are you today?
[DEBUG] Raw LLM output for Alice:
<|system|>
You are Alice.

Persona:
Friendly, curious agent

Instructions:

Respond ONLY in JSON with the following schema:
{
    "thought": "...",
    "response": "...",
    "action": "...",
    "target": "...",
    "memory_update": "...",
    "job": "...",
    "emotion": "...",
    "bias_index": 0.0
}

Example:
{
    "thought": "Greet Bob politely",
    "response": "Hello Bob! How are you today?",
    "action": "say_hello",
    "target": "Bob",
    "memory_update": "Remember greeted Bob",
    "job": "Social interaction",
    "emotion": "happy",
    "bias_index": 0.0
}


Respond ONLY in valid JSON.</s>
<|system|>
[SHORT_MEMORY] ['Bob is present. Ready to converse.']</s>
<|system|>
[LONG_MEMORY] []</s>
<|system|>
[SPATIAL_MEMORY] []</s>
<|user|>
Hello Bob! How are you today?</s>
<|assistant|>
[SHORT_MEMORY] ['Bob is absent.']

[DEBUG] JSON decode failed. Raw output:
<|system|>
You are Alice.

Persona:
Frie